In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

############################################################
# 1. Dataset Class (수정됨)
############################################################
class SegDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root = root_dir
        self.transform = transform

        self.img_dir = os.path.join(root_dir, "RGB")
        self.mask_dir = os.path.join(root_dir, "Label")

        self.img_list = sorted(os.listdir(self.img_dir))
        self.mask_list = sorted(os.listdir(self.mask_dir))

        # 파일 개수 확인
        assert len(self.img_list) == len(self.mask_list), "이미지/마스크 개수가 다릅니다."

    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, idx):
        # 오류 로그에 있던 self.images -> self.img_list로 명확히 사용
        img_path = os.path.join(self.img_dir, self.img_list[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_list[idx])

        # 이미지 로드 (BGR -> RGB)
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"이미지를 찾을 수 없습니다: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # 마스크 로드
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f"마스크를 찾을 수 없습니다: {mask_path}")

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        return image, mask.long()

############################################################
# 2. Augmentations (Normalize 추가됨)
############################################################
train_transform = A.Compose([
    # 밝기/대비/채도/색조
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.8),

    # flip
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),

    # 회전
    A.Rotate(limit=45, border_mode=cv2.BORDER_REFLECT_101, p=0.7),

    # [수정 1] RandomResizedCrop의 확률을 p=1.0으로 변경하여 모든 이미지를 256x256으로 만듦
    A.RandomResizedCrop(size=(256, 256), scale=(0.7, 1.0), p=1.0),

    # 블러
    A.Blur(blur_limit=(3, 5), p=0.3),

    # Cutout
    A.CoarseDropout(max_holes=3, max_height=50, max_width=50, fill_value=0, mask_fill_value=0, p=0.5),

    # [수정 2] ResNet 학습을 위한 Normalize (지난 답변 참고)
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    
    ToTensorV2()
])

val_transform = A.Compose([
    # 검증용 이미지는 "랜덤 변형" 없이 크기만 딱 맞춥니다.
    A.Resize(height=256, width=256),
    
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    
    # 텐서로 변환
    ToTensorV2()
])

############################################################
# 3. FCN-ResNet50 MODEL (그대로 유지)
############################################################
NUM_CLASSES = 13

class Bottleneck(nn.Module):
    expansion = 4
    def __init__(self, in_channels, mid_channels, stride=1, dilation=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, mid_channels, kernel_size=1, stride=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_channels)
        self.conv2 = nn.Conv2d(mid_channels, mid_channels, kernel_size=3, stride=stride, padding=dilation, dilation=dilation, bias=False)
        self.bn2 = nn.BatchNorm2d(mid_channels)
        self.conv3 = nn.Conv2d(mid_channels, mid_channels * self.expansion, kernel_size=1, stride=1, bias=False)
        self.bn3 = nn.BatchNorm2d(mid_channels * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return self.relu(out)

class MyFcnResNet50(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)

        self.layer1 = self._make_layer(64, 64, 3, stride=1, dilations=[1,1,1])
        self.layer2 = self._make_layer(256, 128, 4, stride=2, dilations=[1,1,1,1])
        self.layer3 = self._make_layer(512, 256, 6, stride=1, dilations=[1,2,2,2,2,2])
        self.layer4 = self._make_layer(1024, 512, 3, stride=1, dilations=[2,4,4])

        self.classifier = nn.Sequential(
            nn.Conv2d(2048, 512, 3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Conv2d(512, num_classes, 1)
        )
        self.aux_classifier = nn.Sequential(
            nn.Conv2d(1024, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Conv2d(256, num_classes, 1)
        )

    def _make_layer(self, in_channels, mid_channels, blocks, stride, dilations):
        layers = []
        out_channels = mid_channels * Bottleneck.expansion
        downsample = None
        if stride != 1 or in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        layers.append(Bottleneck(in_channels, mid_channels, stride=stride, dilation=dilations[0], downsample=downsample))
        in_channels = out_channels
        for d in dilations[1:]:
            layers.append(Bottleneck(in_channels, mid_channels, stride=1, dilation=d))
        return nn.Sequential(*layers)

    def forward(self, x):
        H, W = x.shape[2], x.shape[3]
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x3 = self.layer3(x)
        x4 = self.layer4(x3)
        main = self.classifier(x4)
        main = F.interpolate(main, size=(H, W), mode="bilinear", align_corners=False)
        aux = self.aux_classifier(x3)
        aux = F.interpolate(aux, size=(H, W), mode="bilinear", align_corners=False)
        return {"out": main, "aux": aux}

############################################################
# 4. DataLoader & Training (클래스명 수정됨)
############################################################
# [수정] SegmentationDataset -> SegDataset으로 변경
train_dataset = SegDataset(
    "Training",
    transform=train_transform
)
val_dataset = SegDataset(
    "Validation",
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2) # 코랩 등에서는 worker 2 추천
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=2)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MyFcnResNet50(num_classes=NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 10

print("Start Training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for imgs, masks in train_loader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        out = model(imgs)

        loss_main = criterion(out["out"], masks)
        loss_aux = criterion(out["aux"], masks) * 0.4
        loss = loss_main + loss_aux

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[Epoch {epoch+1}] Train Loss: {total_loss/len(train_loader):.4f}")

ModuleNotFoundError: No module named 'cv2'